# Signatures

In [ ]:
import dspy
import os

GEMINI_API_KEY = os.environ.get(key='GEMINI_API_KEY')

dspy.configure(lm=dspy.LM(
    model="gemini/gemini-2.5-flash",
    temperature=0.6|
))

toxicity = dspy.Predict(
    dspy.Signature(
        "comment -> toxic:bool",
        instructions="Mark as toxic if the comment includes insults, haressment, or  or sarcastic derogatory remarks."
    )
)

comment = "You are beautiful"
toxicity(comment=comment).toxic

False

### Sentiment Classification

In [8]:
sentence = "it's a charming and often affecting journey."
classify = dspy.Predict("sentence -> sentiment:bool")
classify(sentence=sentence).sentiment

True

### Summarization

In [13]:
document = """The 21-year-old made seven appearances 
for the Hammers and netted his only goal for them in 
a Europa League qualification round match against 
Andorran side FC Lustrains last season. Lee had two 
loan spells in League One last term, with Blackpool 
and then Colchester United. He scored twice for the 
U's but was unable to save them from relegation.
 The length of Lee's contract with the promoted 
 Tykes has not been revealed. Find all the latest 
 football transfers on our dedicated page."""

summarize = dspy.ChainOfThought("document -> summary")
response = summarize(document=document)
print(response.summary)
print("Reasoning: ",response.reasoning)


21-year-old Lee, who made seven appearances and scored one goal for West Ham, has joined the promoted Tykes. Last season, he had loan spells at Blackpool and Colchester United, scoring twice for Colchester but failing to prevent their relegation. The length of his new contract has not been disclosed.
Reasoning:  The document describes a football player named Lee, his age, his previous club (West Ham), his performance there and during two loan spells (Blackpool and Colchester United), and his transfer to a new club (the 'promoted Tykes'). I will extract these key details to form the summary.


# Class-based DSPy Signatures¶


### Classification

In [14]:
from typing import Literal

class Emotion(dspy.Signature):
    """Classify emotion."""

    sentence: str = dspy.InputField()
    sentiment: Literal['sadness','joy','love','anger','fear','surprise'] = dspy.OutputField()

sentence = "i started feeling a little vulnerable when the giant spotlight started blinding me"  # from dair-ai/emotion

classify = dspy.Predict(Emotion)
classify(sentence=sentence)

Prediction(
    sentiment='fear'
)

In [19]:
class CheckCitationsFaithfulness(dspy.Signature):
    """Verify that the text is based on the provided context"""

    context: str = dspy.InputField(desc="facts here are assumed to be true")
    text: str = dspy.InputField()
    faithfulness: bool = dspy.OutputField()
    evidence: dict[str, list[str]] = dspy.OutputField(desc="Supporting evidence for claims")

context = "The 21-year-old made seven appearances for the Hammers and netted his only goal for them in a Europa League qualification round match against Andorran side FC Lustrains last season. Lee had two loan spells in League One last term, with Blackpool and then Colchester United. He scored twice for the U's but was unable to save them from relegation. The length of Lee's contract with the promoted Tykes has not been revealed. Find all the latest football transfers on our dedicated page."
text = "Lee scored 3 goals for Colchester United."

faithfulness = dspy.ChainOfThought(CheckCitationsFaithfulness)
faithfulness(context=context, text=text)

Prediction(
    reasoning='The text states that Lee scored 3 goals for Colchester United. However, the context explicitly states that "He scored twice for the U\'s", referring to Colchester United. Therefore, the information in the text contradicts the context.',
    faithfulness=False,
    evidence={'Lee scored 3 goals for Colchester United.': ["He scored twice for the U's but was unable to save them from relegation."]}
)

### Multi-modal image classification

In [20]:
class DogPictureSignature(dspy.Signature):
    """Output the dog breed of the dog in the image"""
    image: dspy.Image = dspy.InputField(desc="An image of a dog")
    answer: str = dspy.OutputField(desc="The dog breed of the dog in the image")
    
image_url = "https://picsum.photos/id/237/200/300"
classify = dspy.Predict(DogPictureSignature)
classify(image=dspy.Image.from_url(image_url))

Prediction(
    answer='Labrador Retriever'
)

### Type Resolution in Signatures

In [ ]:
# Working with custom type
from pydantic import BaseModel

class QueryResult(BaseModel):
    text: str
    score: str

signature = dspy.Signature("query: str -> result: QueryResult")

class MyContainer:
    class Query(BaseModel):
        text: str
    class Score(BaseModel):
        score: float
    
signature = dspy.Signature("query: MyContainer.Query -> score: MyContainer.Score")
